# DCASE 2026 Task 1 — Heterogeneous Audio Classification
## Baseline (HATR) — VSCode 실행 노트북 (Windows + RTX 3060)

**실행 전 체크리스트**
- VSCode에 **Jupyter 확장** (by Microsoft) 설치 확인
- CUDA 12.x 드라이버 설치 확인 (`nvidia-smi` 로 확인)
- 데이터셋 다운로드 필요: [BSD10k-v1.2](https://zenodo.org/records/17233904) / [BSD35k-CS](https://zenodo.org/records/19187099)

**디렉토리 구조**
```
DCASE_baseline/                        ← 최상위 루트 (PROJECT_DIR)
├── data/                              ← 데이터셋 폴더
│   ├── metadata/
│   │   ├── BSD10k_metadata.csv
│   │   ├── BSD35k-CS_metadata.csv
│   │   └── BST_description.csv
│   └── features/
│       ├── clap_audio_embeddings/
│       └── clap_text_embeddings/
└── dcase2026_task1_baseline/          ← git clone 위치 (코드 루트)
    ├── train_test.py
    ├── config.yaml
    └── ...
```

---

## 0. GPU 확인

In [3]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  CUDA를 찾지 못했습니다. PyTorch CUDA 버전 설치를 확인하세요.')
    print('    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')

CUDA available: True
GPU: NVIDIA GeForce RTX 3060
VRAM: 8.6 GB


## 1. 패키지 설치

In [4]:
# 가상환경이 활성화된 상태에서 노트북을 실행하고 있는지 확인
import sys
print('Python 경로:', sys.executable)
# 출력에 .venv 경로가 포함되어 있어야 정상
# 예: C:\Users\yourname\DCASE_baseline\.venv\Scripts\python.exe

# 패키지 설치는 터미널에서 이미 완료했으므로 import 가능 여부만 확인
import torch, numpy, pandas, sklearn, yaml, matplotlib
print('모든 패키지 import 성공')

Python 경로: c:\Users\solok\Desktop\Dcase baseline\.venv\Scripts\python.exe
모든 패키지 import 성공


## 2. 경로 설정

**⚠️ `PROJECT_DIR`을 본인의 최상위 폴더 경로로 수정하세요.**

예시:
- `C:\Users\yourname\DCASE_baseline`
- `D:\projects\DCASE_baseline`

In [6]:
import os

# ↓↓↓ 여기만 본인 경로로 수정 ↓↓↓
PROJECT_DIR = r"C:\Users\solok\Desktop\Dcase baseline"  # 최상위 루트 (data/ 폴더가 있는 곳)
# ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑

CODE_DIR = os.path.join(PROJECT_DIR, 'dcase2026_task1_baseline')  # git clone 위치
DATA_ROOT = os.path.join(PROJECT_DIR, 'data')                     # 데이터셋 위치

# 폴더 자동 생성
os.makedirs(CODE_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

print('최상위 루트  :', PROJECT_DIR)
print('코드 위치    :', CODE_DIR)
print('데이터 위치  :', DATA_ROOT)

최상위 루트  : C:\Users\solok\Desktop\Dcase baseline
코드 위치    : C:\Users\solok\Desktop\Dcase baseline\dcase2026_task1_baseline
데이터 위치  : C:\Users\solok\Desktop\Dcase baseline\data


## 3. 베이스라인 코드 세팅

> **방법 A**: GitHub에서 클론 (권장)  
> **방법 B**: 이미 clone 해둔 경우 스킵

In [ ]:
# 베이스라인 코드 세팅 (idempotent)
# 이미 clone되어 있으면 자동 스킵
os.chdir(PROJECT_DIR)
if not os.path.isdir(CODE_DIR) or not os.path.isfile(os.path.join(CODE_DIR, 'train_test.py')):
    print('clone 시작...')
    !git clone https://github.com/MTG/dcase2026_task1_baseline.git
else:
    print(f'이미 존재함 → 스킵: {CODE_DIR}')

os.chdir(CODE_DIR)
print('작업 디렉토리:', os.getcwd())

## 4. config.yaml 경로 설정

`DATA_ROOT`는 위 셀에서 자동으로 설정되었습니다.  
`ACTIVE_DATASET`만 원하는 데이터셋으로 바꾸세요.

In [8]:
ACTIVE_DATASET = 'BSD10k-v1.2'  # 'BSD10k-v1.2' 또는 'BSD35k-CS'

# Windows 경로의 역슬래시를 슬래시로 변환 (yaml 파싱 안전)
data_root_str = DATA_ROOT.replace('\\', '/')

config_content = f"""# INPUT PATHS
active_dataset: {ACTIVE_DATASET}
datasets:
  BSD10k-v1.2:
    metadata_csv: "{data_root_str}/metadata/BSD10k_metadata.csv"
    audio_emb_folder: "{data_root_str}/features/clap_audio_embeddings"
    text_emb_folder: "{data_root_str}/features/clap_text_embeddings"
    class_names: "{data_root_str}/metadata/BST_description.csv"

  BSD35k-CS:
    metadata_csv: "{data_root_str}/metadata/BSD35k-CS_metadata.csv"
    audio_emb_folder: "{data_root_str}/features/clap_audio_embeddings"
    text_emb_folder: "{data_root_str}/features/clap_text_embeddings"
    class_names: "{data_root_str}/metadata/BST_description.csv"

# OUTPUT PATHS
output_path: "data"
processed_dataset_csv: "processed_dataset.csv"
class_dict_json: "class_dict.json"
top_class_dict_json: "top_class_dict.json"
top_class_subclass_dict_json: "top_class_subclass_dict.json"
"""

with open('config.yaml', 'w') as f:
    f.write(config_content)

print('config.yaml 저장 완료')
with open('config.yaml') as f:
    print(f.read())

config.yaml 저장 완료
# INPUT PATHS
active_dataset: BSD10k-v1.2
datasets:
  BSD10k-v1.2:
    metadata_csv: "C:/Users/solok/Desktop/Dcase baseline/data/metadata/BSD10k_metadata.csv"
    audio_emb_folder: "C:/Users/solok/Desktop/Dcase baseline/data/features/clap_audio_embeddings"
    text_emb_folder: "C:/Users/solok/Desktop/Dcase baseline/data/features/clap_text_embeddings"
    class_names: "C:/Users/solok/Desktop/Dcase baseline/data/metadata/BST_description.csv"

  BSD35k-CS:
    metadata_csv: "C:/Users/solok/Desktop/Dcase baseline/data/metadata/BSD35k-CS_metadata.csv"
    audio_emb_folder: "C:/Users/solok/Desktop/Dcase baseline/data/features/clap_audio_embeddings"
    text_emb_folder: "C:/Users/solok/Desktop/Dcase baseline/data/features/clap_text_embeddings"
    class_names: "C:/Users/solok/Desktop/Dcase baseline/data/metadata/BST_description.csv"

# OUTPUT PATHS
output_path: "data"
processed_dataset_csv: "processed_dataset.csv"
class_dict_json: "class_dict.json"
top_class_dict_json: "top_

## 5. 데이터셋 경로 확인

실제 파일이 있는지 확인합니다.

In [9]:
import os

checks = [
    os.path.join(DATA_ROOT, 'metadata', 'BSD10k_metadata.csv'),
    os.path.join(DATA_ROOT, 'features', 'clap_audio_embeddings'),
    os.path.join(DATA_ROOT, 'features', 'clap_text_embeddings'),
    os.path.join(DATA_ROOT, 'metadata', 'BST_description.csv'),
]

all_ok = True
for path in checks:
    exists = os.path.exists(path)
    status = '✅' if exists else '❌ 없음!'
    print(f'{status}  {path}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n모든 경로 확인 완료. 다음 셀을 실행하세요.')
else:
    print('\n❌ 위 경로에 파일이 없습니다. 데이터셋을 올바른 위치에 배치했는지 확인하세요.')

❌ 없음!  C:\Users\solok\Desktop\Dcase baseline\data\metadata\BSD10k_metadata.csv
✅  C:\Users\solok\Desktop\Dcase baseline\data\features\clap_audio_embeddings
✅  C:\Users\solok\Desktop\Dcase baseline\data\features\clap_text_embeddings
❌ 없음!  C:\Users\solok\Desktop\Dcase baseline\data\metadata\BST_description.csv

❌ 위 경로에 파일이 없습니다. 데이터셋을 올바른 위치에 배치했는지 확인하세요.


In [ ]:
# metadata 폴더 파일 목록 확인 (디렉토리 없어도 안전)
base = os.path.join(DATA_ROOT, 'metadata')
print('metadata 파일 목록:')
if os.path.isdir(base):
    for f in os.listdir(base):
        print(f'  [{f}]')
else:
    print(f'  (디렉토리 없음: {base})')

## 6. train_test.py 버그 수정 패치

원본 코드에 `config.yaml`에 없는 키(`color_dict_path`)를 참조하는 버그가 있어 패치합니다.

In [ ]:
# train_test.py 패치
with open('train_test.py', 'r') as f:
    code = f.read()

code = code.replace(
    'color_dict_path = get_subconfig("color_dict_path")\n',
    '# color_dict_path removed (not in config)\n'
)
code = code.replace(
    'top_color_dict_path = get_subconfig("top_color_dict_path")\n',
    '# top_color_dict_path removed (not in config)\n'
)

with open('train_test.py', 'w') as f:
    f.write(code)

print('train_test.py 패치 완료')

# evaluate.py 패치
with open('evaluate.py', 'r') as f:
    code = f.read()

code = code.replace(
    'color_dict_path = get_subconfig("color_dict_path")\n',
    '# color_dict_path removed\n'
)
code = code.replace(
    'top_color_dict_path = get_subconfig("top_color_dict_path")\n',
    '# top_color_dict_path removed\n'
)

with open('evaluate.py', 'w') as f:
    f.write(code)

print('evaluate.py 패치 완료')

## 7. DataLoader num_workers 패치 (Windows 호환)

Windows에서 Jupyter 환경은 `if __name__ == '__main__'` 가드 없이 실행되므로  
`num_workers=0`으로 설정해야 멀티프로세싱 오류가 발생하지 않습니다.

In [ ]:
for fname in ['train_test.py', 'evaluate.py']:
    with open(fname, 'r') as f:
        code = f.read()
    # Windows Jupyter 환경: num_workers=0 이 안전
    code = code.replace('num_workers=4', 'num_workers=0')
    code = code.replace('num_workers=2', 'num_workers=0')
    with open(fname, 'w') as f:
        f.write(code)

print('num_workers 패치 완료 (→ 0, Windows 호환)')

## 8. 데이터셋 빌드

임베딩 파일 경로를 스캔하고 `processed_dataset.csv` 생성

In [ ]:
# 데이터셋 빌드 (idempotent: processed_dataset.csv 있으면 스킵)
import os, sys, subprocess
if os.path.isfile('data/processed_dataset.csv'):
    print('processed_dataset.csv 존재 → 빌드 스킵')
else:
    ret = subprocess.run([sys.executable, 'build_dataset.py'], check=False)
    assert ret.returncode == 0, f'build_dataset.py 실패 (exit code {ret.returncode})'

In [ ]:
# 빌드 결과 확인
import pandas as pd
df = pd.read_csv('data/processed_dataset.csv')
print(f'총 샘플 수: {len(df)}')
print(f'클래스 수: {df["class"].nunique()}')
print('\n클래스 분포:')
print(df['class'].value_counts().to_string())

---
## 9. EXP-010 — 진정한 baseline (train-only conf 필터 + GroupKFold)

**중요한 방법론 수정 2가지를 모두 반영한 새 baseline:**

### Fix 1: confidence 필터를 train set에만 적용 (BUG FIX)
- **문제:** 기존 코드는 `conf>=4` 필터를 fold split *전*에 전체 데이터에 적용 → val/test도 깨끗한 샘플만 평가 → H-Acc 부풀려짐
- **수정:** fold split *후* train만 필터, val/test는 모든 샘플 유지 → 실제 운영 환경 반영
- 이전 EXP-001 ~ EXP-008 점수는 모두 inflated → 비교 기준에서 폐기

### Fix 2: StratifiedGroupKFold by uploader
- 1,806 uploaders 중 77.6%가 단일 클래스만 업로드 → 기존 random KFold는 uploader leakage 발생
- uploader 단위 분리로 누설 차단

### 예상 결과
- 두 fix 결합으로 H-Acc는 EXP-008(85.29%)보다 **상당히 낮게** 나올 가능성 높음 (예상 70~80% 범위)
- **이 점수가 진짜 private LB proxy**. 이후 모든 실험은 이 새 baseline과 비교
- RTX 3060 기준 1~2시간 예상

In [ ]:
# EXP-010: EXP-008 best config + StratifiedGroupKFold by uploader
import os, sys, subprocess
print('cwd:', os.getcwd())
assert os.path.basename(os.getcwd()) == 'dcase2026_task1_baseline', \
    "CODE_DIR로 이동 필요 (이전 셀에서 자동 설정됨)"

cmd = [
    sys.executable, 'train_test.py',
    '--exp_name', 'exp_010_groupkfold',
    '--modes', 'both',
    '--conf_threshold', '4',
    '--hier_loss',
    '--lambda_top', '0.3', '--lambda_contr', '0.1', '--tau', '0.07',
    '--hidden_size', '256', '--dropout', '0.1',
    '--epochs', '100', '--batch_size', '64', '--lr', '0.001',
    '--k_folds', '5', '--class_weights', 'balanced',
    '--fold_strategy', 'stratified_group',
]
print('CMD:', ' '.join(cmd))
# 학습 진행상황을 실시간으로 보려면 stdout=None (default)로 두고 셀 출력에 직접 흘려보냄
ret = subprocess.run(cmd, check=False)
print(f'\n[exit code] {ret.returncode}')
assert ret.returncode == 0, f'EXP-010 학습 실패 (exit code {ret.returncode})'

In [ ]:
# EXP-010 결과 요약 + experiments JSON 저장
import json, os, subprocess
import numpy as np

summary_path = 'model_output/exp_010_groupkfold/summary.json'
with open(summary_path) as f:
    s = json.load(f)

both_runs = [r for r in s['fold_metrics'] if r['mode'] == 'both']
keys = ['accuracy','top_accuracy','macro_accuracy','macro_top_accuracy',
        'hierarchical_accuracy','hierarchical_precision','hierarchical_recall','hierarchical_f1']

print('=== EXP-010 (TRUE baseline: train-only conf filter + StratifiedGroupKFold) ===')
fold_avg = {}
for k in keys:
    vals = [r[k] for r in both_runs]
    fold_avg[k] = float(np.mean(vals))
    print(f'  {k:30s}: {np.mean(vals):.2f}% +- {np.std(vals):.2f}%')

# 이전 inflated 점수 대비
EXP008_INFLATED = 85.29
delta_vs_008 = fold_avg['hierarchical_accuracy'] - EXP008_INFLATED
print(f"\nvs EXP-008 inflated (85.29%): {delta_vs_008:+.2f}%")
print("  (참고: EXP-008은 conf 필터를 test에도 적용 + random KFold uploader leakage")
print("   두 결함이 합쳐진 inflated 점수. 본 EXP-010이 진정한 baseline.)")

# experiments/ JSON 저장
os.makedirs('../experiments', exist_ok=True)
try:
    git_commit = subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip()
except Exception:
    git_commit = 'unknown'

import datetime
record = {
    'exp_id': 'exp_010',
    'branch': 'main',
    'description': 'TRUE baseline: train-only conf filter + StratifiedGroupKFold by uploader',
    'methodology_fixes': [
        'conf filter applied to TRAIN ONLY (not val/test) — Fix 1',
        'StratifiedGroupKFold by uploader (uploader leakage 차단) — Fix 2',
    ],
    'config': s['config'],
    'results': {
        'fold_avg': {
            'macro_acc_2nd': fold_avg['macro_accuracy']/100,
            'macro_acc_top': fold_avg['macro_top_accuracy']/100,
            'hierarchical_acc': fold_avg['hierarchical_accuracy']/100,
            'hierarchical_f1': fold_avg['hierarchical_f1']/100,
        },
        'fold_details': both_runs,
    },
    'vs_exp008_inflated_h_acc': f'{delta_vs_008:+.2f}%',
    'note': 'EXP-001~009 점수는 모두 inflated. 본 실험이 새 baseline.',
    'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M'),
    'git_commit': git_commit,
}

out_json = '../experiments/exp_010_true_baseline.json'
with open(out_json, 'w') as f:
    json.dump(record, f, indent=2, default=str)
print(f"\nSaved: {out_json}")
print(f"\n>>> 새 baseline H-Acc: {fold_avg['hierarchical_accuracy']:.2f}% <<<")
print(f"    이후 모든 실험은 이 값과 비교하세요.")

---
## 알려진 이슈 & 해결법

| 증상 | 원인 | 해결 |
|------|------|------|
| `KeyError: 'color_dict_path'` | config.yaml에 없는 키 참조 | 셀 6 패치 적용 |
| `RuntimeError: DataLoader worker` 오류 | Windows 멀티프로세싱 제한 | 셀 7 패치 (num_workers=0) |
| `CUDA out of memory` | batch_size 너무 큼 | `BATCH_SIZE=32` 로 줄이기 |
| `CUDA available: False` | PyTorch CPU 버전 설치됨 | `pip install torch ... --index-url https://download.pytorch.org/whl/cu121` |
| `Missing audio embedding` 많이 출력 | 임베딩 경로 불일치 | `DATA_ROOT` 경로 재확인, 셀 4 재실행 |
| `ModuleNotFoundError` | 작업 디렉토리가 CODE_DIR이 아님 | 셀 3-B 재실행 후 `os.chdir(CODE_DIR)` |

---
*DCASE 2026 Task 1 — Good luck!*